In [2]:
import tensorflow as tf
from tensorflow.keras import layers
print(tf.__version__)

2.21.0


In [3]:
x = tf.constant([
    [
        [0.1, 0.2, 0.3, 0.4],  # The
        [0.5, 0.6, 0.7, 0.8],  # animal
        [0.9, 1.0, 1.1, 1.2],  # crossed
        [1.3, 1.4, 1.5, 1.6]   # street
    ]
], dtype=tf.float32)

print("Shape:", x.shape)

Shape: (1, 4, 4)


In [4]:
attention_layer = layers.MultiHeadAttention(
    num_heads=2,
    key_dim=4
)

In [5]:
output = attention_layer(
    query=x,
    value=x,
    key=x
)

print("Output shape:", output.shape)

Output shape: (1, 4, 4)


In [6]:
output, attention_scores = attention_layer(
    query=x,
    value=x,
    key=x,
    return_attention_scores=True
)

print("Output shape:", output.shape)
print("Attention scores shape:", attention_scores.shape)

Output shape: (1, 4, 4)
Attention scores shape: (1, 2, 4, 4)


In [7]:
print(attention_scores[0, 0].numpy())

[[0.25676993 0.25220272 0.24771677 0.24331057]
 [0.25782853 0.2525375  0.24735504 0.24227892]
 [0.258889   0.25287026 0.24699144 0.2412493 ]
 [0.25995126 0.25320098 0.24662597 0.24022171]]


In [8]:
row_sums = tf.reduce_sum(attention_scores, axis=-1)

print(row_sums.numpy())

[[[1.         1.         1.         0.99999994]
  [0.99999994 1.         1.         1.        ]]]


In [9]:
print(output.numpy())

[[[0.3489423  0.70804477 0.01092379 0.18857414]
  [0.34850973 0.7076739  0.0112593  0.18945414]
  [0.34807712 0.70730305 0.01159484 0.1903342 ]
  [0.3476446  0.7069323  0.01193024 0.19121408]]]


In [10]:
#Now let's make this visual
import pandas as pd

tokens = ["The", "animal", "crossed", "street"]

attention_matrix = attention_scores[0, 0].numpy()

df = pd.DataFrame(
    attention_matrix,
    index=tokens,
    columns=tokens
)

print(df)

              The    animal   crossed    street
The      0.256770  0.252203  0.247717  0.243311
animal   0.257829  0.252537  0.247355  0.242279
crossed  0.258889  0.252870  0.246991  0.241249
street   0.259951  0.253201  0.246626  0.240222


In [21]:
class TransformerBlock(layers.Layer):

    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

    def call(self, x):

        # 1. Self-Attention
        attention_output = self.attention(
            query=x,
            key=x,
            value=x
        )

        # 2. Residual + LayerNorm
        normalized_1 = self.norm1(
            x + attention_output
        )

        # 3. Feed-Forward Network
        ffn_output = self.ffn(normalized_1)

        # 4. Residual + LayerNorm
        final_output = self.norm2(
            normalized_1 + ffn_output
        )

        return (
            attention_output,
            normalized_1,
            ffn_output,
            final_output
        )

In [22]:
transformer_block = TransformerBlock(
    embed_dim=4,
    num_heads=2,
    ff_dim=16
)

In [23]:
attention_output, normalized_1, ffn_output, final_output = transformer_block(x)

In [24]:
print("Input:")
print(x.numpy())

print("\nAttention Output:")
print(attention_output.numpy())

print("\nAfter Residual + LayerNorm:")
print(normalized_1.numpy())

print("\nFFN Output:")
print(ffn_output.numpy())

print("\nFinal Output:")
print(final_output.numpy())

Input:
[[[0.1 0.2 0.3 0.4]
  [0.5 0.6 0.7 0.8]
  [0.9 1.  1.1 1.2]
  [1.3 1.4 1.5 1.6]]]

Attention Output:
[[[0.1041148  1.0085791  1.1052291  0.51246774]
  [0.12821311 1.0068537  1.1655142  0.50860155]
  [0.1512546  1.0070692  1.2246295  0.5060583 ]
  [0.17276415 1.0098916  1.2818857  0.50530493]]]

After Residual + LayerNorm:
[[[-1.5947775   0.604172    1.0346732  -0.04406822]
  [-1.563053    0.54950595  1.107868   -0.09432101]
  [-1.5298212   0.4982357   1.1720386  -0.14045215]
  [-1.4972124   0.4523282   1.2262006  -0.18131447]]]

FFN Output:
[[[ 0.2963488  -0.32886398  0.25600594 -0.49530533]
  [ 0.33176166 -0.33627963  0.25926217 -0.46682906]
  [ 0.36340117 -0.34233972  0.26152942 -0.4396335 ]
  [ 0.39063466 -0.347173    0.26293048 -0.41489118]]]

Final Output:
[[[-1.2786928   0.35671297  1.411873   -0.48989308]
  [-1.2188257   0.2754117   1.4690325  -0.5256185 ]
  [-1.1603733   0.200907    1.516228   -0.55676156]
  [-1.1060994   0.13554475  1.5536836  -0.583129  ]]]
